[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/15_mlp.ipynb)

# 🟠 中等：SwiGLU 多层感知机

实现现代大语言模型（如 LLaMA）中使用的 **SwiGLU MLP**（前馈网络）。

$$\text{SwiGLU}(x) = \text{down\_proj}\big(\text{SiLU}(\text{gate\_proj}(x)) \odot \text{up\_proj}(x)\big)$$

其中 $\text{SiLU}(x) = x \cdot \sigma(x)$

### 函数签名
```python
class SwiGLUMLP(nn.Module):
    def __init__(self, d_model: int, d_ff: int): ...
    def forward(self, x: torch.Tensor) -> torch.Tensor: ...
```

### 要求
- 继承自 `nn.Module`
- `self.gate_proj`：`nn.Linear(d_model, d_ff)`
- `self.up_proj`：`nn.Linear(d_model, d_ff)`
- `self.down_proj`：`nn.Linear(d_ff, d_model)`
- 激活函数：**SiLU**（又称 Swish）—— `F.silu` 或实现为 `x * torch.sigmoid(x)`

### 为什么用 SwiGLU？
与经典的 `Linear → ReLU/GELU → Linear` 前馈网络不同，SwiGLU 使用**门控机制**：
门控投影控制信息流，而上投影提供内容。
在实践中，SwiGLU 始终优于标准前馈网络（PaLM、LLaMA、Mistral 均采用此结构）。

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# ✏️ 在此实现你的代码

class SwiGLUMLP(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        pass  # 初始化 gate_proj, up_proj, down_proj

    def forward(self, x):
        pass  # down_proj(silu(gate_proj(x)) * up_proj(x))

SwiGLU MLP 引入 SiLU 激活函数和 gate 投影得到门控机制，控制上投影部分信息。

传统的 MLP 只有两层线性映射（$2 \times d_{model} \times d_{ff}$ 参数），而 SwiGLU 包含三层线性映射（$3 \times d_{model} \times d_{ff}$ 参数）。注意：为了保持计算量一致，使用 SwiGLU 时通常会将 d_ff 设置得比传统 ReLU MLP 略小一些（例如 LLaMA 采用 $\frac{2}{3} \times 4d_{model} \approx 2.67d_{model}$）。\
在大多数现代 LLM 实现中，这些线性层通常设置 bias=False，这有助于训练稳定性并略微提高推理效率。

SiLU（Sigmoid Linear Unit）激活函数，也被称为 Swish 激活函数。它的数学表达式如下：$$f(x) = x \cdot \sigma(x)$$其中，$\sigma(x)$ 是标准的 Sigmoid 函数：$$\sigma(x) = \frac{1}{1 + e^{-x}}$$将两者结合，完整的公式可以写作：$$f(x) = \frac{x}{1 + e^{-x}}$$
主要特性
- 平滑性：与 ReLU 不同，SiLU 在 $x=0$ 处是二阶可导的，这有助于优化过程中的梯度流动。
- 非单调性：在 $x < 0$ 的区域，SiLU 并不是完全为 0 或单调递增，而是在大约 $x \approx -1.28$ 处有一个全局最小值（约为 $-0.28$）。
- 有下界、无上界：这使得它在正区间保留了 ReLU 的优势（防止梯度消失），而在负区间通过微小的负值保留了一些神经元的活性。

In [ ]:
class SwiGLUMLP(nn.Module):
    def __init__(self, d_model: int, d_ff: int):
        """
        初始化 SwiGLU 多层感知机。
        
        Args:
            d_model: 输入和输出的嵌入维度
            d_ff: 中间隐藏层的维度（通常是 4 * d_model，或者在 LLaMA 中约为 8/3 * d_model）
        """
        super().__init__()
        # 门控分支：用于生成门控信号
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        # 上投影分支：用于提取特征内容
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        # 下投影分支：将维度映射回 d_model
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        前向传播计算。
        公式：down_proj(SiLU(gate_proj(x)) * up_proj(x))
        """

        # [batch_size, seq_len, d_model] -> [batch_size, seq_len, d_ff]
        # 计算门控部分并应用 SiLU 激活函数
        gate = F.silu(self.gate_proj(x))
        
        # 计算上投影部分
        up = self.up_proj(x)
        
        # 逐元素相乘（门控机制）
        # 这一步是 SwiGLU 的核心：gate 控制了 up 中哪些信息可以通过
        activated_features = gate * up # 这里是每个值相乘，不是 matmul 点乘
        
        # 最后投影回原始维度
        return self.down_proj(activated_features)

In [ ]:
# 🧪 调试
mlp = SwiGLUMLP(d_model=64, d_ff=128)
x = torch.randn(2, 8, 64)
out = mlp(x)
print("输出形状:", out.shape)  # (2, 8, 64)
print("参数量:", sum(p.numel() for p in mlp.parameters()))

In [ ]:
# ✅ 提交
from torch_judge import check
check('mlp')